In [ ]:
import os
import sys

# Ensure repository root is on Python path
sys.path.insert(0, os.path.abspath("."))

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
import matplotlib.pyplot as plt

# Import from our modular package
from src.features import build_features, make_target, select_top_features

CSV_PATH = "data/XAU_1m_data.csv"
OUT_DIR = "model_output"
os.makedirs(OUT_DIR, exist_ok=True)
print("Features setup ready.")


In [ ]:
# ---------- 1. Load 1-min Data and Resample to 1-Hour ----------
print(f"Loading {CSV_PATH} ...")
raw = pd.read_csv(CSV_PATH, parse_dates=["Date"]).set_index("Date").sort_index()

df_1h = (
    raw.resample("1h")
    .agg({"Open": "first", "High": "max", "Low": "min", "Close": "last", "Volume": "sum"})
    .dropna()
)
print(f"Hourly dataset: {len(df_1h):,} bars ({df_1h.index.min()} to {df_1h.index.max()})")


In [ ]:
# ---------- 2. Build Stationary Features ----------
print("Building stationary features via src.features.build_features() ...")
feats = build_features(df_1h)
y_reg = make_target(df_1h, horizon=1, kind="regression")
y_clf = make_target(df_1h, horizon=1, kind="classification")

print(f"Generated {feats.shape[1]} features across {len(feats):,} bars.")
print("\nFeature categories created:")
print("- Multi-horizon returns: ret_1 to ret_34, ret_240")
print("- Autoregressive past 1-bar returns: ret_lag_1h to ret_lag_10h (shift-based)")
print("- Relative MA distance: px_over_ma_5 to px_over_ma_200 (Close / MA - 1)")
print("- Volatility & Normalized ATR: vol_5 to vol_50, atr_pct_14, atr_pct_50")
print("- Normalized Bollinger Bands: bb_pct_b, bb_width")
print("- Wilder's RSI: rsi_7, rsi_14, rsi_21")
print("- Normalized MACD: macd_norm, macd_signal_norm, macd_hist_norm")
print("- Volume z-score & ratios: vol_zscore_20, vol_ratio_ma_5, etc.")
print("- Candle geometry: candle_body, candle_upper_wick, candle_lower_wick")
print("- Session & Cyclical Time: hour_sin, hour_cos, dow_sin, dow_cos, is_market_gap")


In [ ]:
# ---------- 3. Verify Stationarity & Clean Data ----------
# Combine features and regression target
data = feats.join(y_reg.rename("target")).dropna()
X = data.drop(columns=["target"])
y = data["target"]

print(f"Clean samples after dropping warmup rolling windows: {len(data):,}")

# Check that NO raw absolute price columns exist
raw_price_cols = [c for c in X.columns if c in ["Close", "High", "Low", "Open", "ma_20", "bb_upper", "bb_lower"]]
if len(raw_price_cols) == 0:
    print("✅ Stationarity check passed: No raw dollar price levels found in feature set.")
else:
    print("⚠️ WARNING: Found raw price columns:", raw_price_cols)

print("\nSample feature summary:")
print(X[["ret_1", "ret_lag_1h", "px_over_ma_20", "atr_pct_14", "rsi_14", "bb_pct_b", "hour_sin"]].describe().round(4))


In [ ]:
# ---------- 4. Feature Selection Demo (Strictly on Train Split) ----------
# To prevent lookahead bias (data leakage), feature selection MUST be fitted
# strictly on the training partition, NOT on the whole dataset!
n_total = len(X)
train_size = int(n_total * 0.72)
X_train = X.iloc[:train_size]
y_train = y.iloc[:train_size]

print(f"Running feature selection on training split ({len(X_train):,} samples) ...")
top_features, importances = select_top_features(X_train, y_train, n=20)

print(f"\nTop 15 Most Informative Features (strictly from train set):")
for rank, feat in enumerate(top_features[:15], 1):
    print(f" {rank:2d}. {feat:<22} (Importance: {importances[feat]:.4f})")


In [ ]:
# ---------- 5. Visualize Top Feature Importances ----------
top_imp = importances.head(15).iloc[::-1]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_imp.index, top_imp.values, color="#3498db", edgecolor="#217dbb")
ax.set_title("Top 15 Feature Importances (Random Forest on Train Set)", fontsize=12, fontweight='bold')
ax.set_xlabel("Relative Importance")
ax.grid(True, alpha=0.3, axis="x")

plt.tight_layout()
feat_plot_path = os.path.join(OUT_DIR, "feature_importances_demo.png")
plt.savefig(feat_plot_path, dpi=200)
plt.show()
print(f"Saved feature importance plot to {feat_plot_path}")
